# 🤖 Building AI Agents - From LLMs to Autonomous Systems!

## 🤔 What Are Agents?

An **AI Agent** is an LLM that can **reason**, **plan**, and **take actions** by using tools. Instead of just generating text, agents can interact with the real world!

**The Evolution:**
- 📝 LLM alone → Can only generate text
- 🧠 LLM + Memory → Can remember past conversations
- 🔧 LLM + Tools → Can take actions (search, calculate, call APIs)
- 🤖 LLM + Memory + Tools + Reasoning = **Agent!**

---

### 🌟 Why Agents Are Revolutionary:

| Plain LLM | Agent |
|-----------|-------|
| 🔤 Generates text only | 🔧 Can use tools & APIs |
| ❌ No memory between calls | ✅ Remembers conversation context |
| 🚫 Can't take actions | 🎯 Can search, calculate, retrieve data |
| 📏 Limited to training data | 🌐 Can access live information |

---

### 🎓 What You'll Learn:

1. 💬 **Memory** - Give your LLM conversation history
2. 🔧 **Tool Calling** - Let the LLM use functions
3. 🔄 **The ReAct Pattern** - Reason + Act in a loop
4. 🏗️ **LangGraph Agents** - Build agents with frameworks
5. 🧩 **Multi-Tool Agents** - Combine multiple capabilities
6. 🚀 **Advanced Patterns** - Structured output, RAG agents

All examples use **Mistral AI** models! 🇫🇷

Let's dive in! 🚀

> **📝 Note:** You'll need a `MISTRAL_API_KEY` to follow along.
>
> **🔑 Get Your Free API Key:**
> 1. Go to [Mistral AI Console](https://console.mistral.ai/home)
> 2. Sign up or log in
> 3. Navigate to "API Keys" section
> 4. Create a new API key
> 5. Copy your key and keep it safe! 🔒

## 🔑 Step 1: Environment Setup

First, let's install the dependencies and set up our API key! 🔒

In [207]:
import os
import time

os.environ["MISTRAL_API_KEY"] = "EIdRWsLoyAH6ATXuYgHShsVd4n9Z4JPl"

print("✅ API key configured!")


✅ API key configured!


In [208]:
from langchain_mistralai import ChatMistralAI

# Initialize our base model - we'll reuse this throughout the notebook
model = ChatMistralAI(model="mistral-small-latest", temperature=0)

# Quick test
response = model.invoke("Say hello in one sentence!")
print("🎉 Model is working!")
print(f"💬 {response.content}")


🎉 Model is working!
💬 "Hello there!"


---

# 🧠 Part 1: Memory - Giving Your LLM a Brain

LLMs are **stateless** — they have no idea about previous interactions. Every API call starts from scratch!

```
Without Memory:              With Memory:
┌─────────────┐              ┌─────────────┐
│ User: Hi!   │              │ User: Hi!   │
│ AI: Hello!  │              │ AI: Hello!  │
└─────────────┘              │ User: Name? │
┌─────────────┐              │ AI: I'm...  │
│ User: Name? │              │ User: ???   │
│ AI: ???     │ ← Forgot!   │ AI: Knows!  │ ← Remembers!
└─────────────┘              └─────────────┘
```

**Memory = passing conversation history back to the LLM on each call.**

### 🔍 The Problem: LLMs Are Stateless

Let's see this in action — the model forgets between calls:

In [209]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# First call
response1 = model.invoke("My name is Alice and I love Python programming.")
print("📩 Call 1:", response1.content)

print("\n" + "="*60 + "\n")

# Second call — the model has NO idea about the first call!
response2 = model.invoke("What is my name and what do I love?")
print("📩 Call 2:", response2.content)
print("\n❌ The model forgot everything from the first call!")


📩 Call 1: That's awesome, Alice! 🎉 Python is such a versatile and beginner-friendly language, and it's great to hear you're passionate about it.

If you'd like, I can help you with:
- **Learning Python** (concepts, best practices, or project ideas)
- **Debugging code** (share snippets, and I’ll help!)
- **Exploring libraries** (e.g., Django, Flask, NumPy, Pandas, etc.)
- **Career advice** (how to level up your Python skills for jobs)

What would you like to focus on today? 😊

*(P.S. If you have a fun Python project you're working on, I’d love to hear about it!)*


📩 Call 2: I don’t have access to personal information about you unless you’ve shared it with me during our conversation. If you’d like, you can tell me your name and what you love, and I’ll remember it for our chat! 😊

For example, you could say:
*"My name is [Your Name], and I love [Your Passion]!"*

Then I can respond with something like:
*"Nice to meet you, [Your Name]! That’s awesome that you love [Your Passion]—I’d love 

### ✅ The Solution: Pass Conversation History

We manually pass all previous messages each time:

In [210]:
# Build conversation history manually
conversation = [
    SystemMessage(content="You are a friendly assistant. Remember everything the user tells you."),
    HumanMessage(content="My name is Alice and I love Python programming."),
    AIMessage(content="Nice to meet you, Alice! Python is a great language."),
    HumanMessage(content="What is my name and what do I love?"),
]

response = model.invoke(conversation)
print("📩 With history:", response.content)
print("\n✅ The model remembers because we passed the full conversation!")


📩 With history: Your name is Alice and you love Python programming.

✅ The model remembers because we passed the full conversation!


### 🏗️ Building a Simple Memory Manager

Let's create a reusable `ChatBot` class with memory:

In [211]:

from langchain_core.messages import BaseMessage


class ChatBot:
    """A simple chatbot with conversation memory."""

    def __init__(self, system_prompt: str = "You are a helpful assistant.") -> None:
        """Initialize the chatbot with a system prompt and an empty history."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.history: list[BaseMessage] = [
            SystemMessage(content=system_prompt),
        ]

    def chat(self, user_message: str) -> str | list:
        """Send a message and get a response, maintaining history."""
        # Add user message to history
        self.history.append(HumanMessage(content=user_message))

        # Call the LLM with FULL history
        response = self.model.invoke(self.history)

        # Add AI response to history
        self.history.append(AIMessage(content=response.content))

        return response.content

    def get_history_length(self) -> int:
        """Get the number of messages in the conversation history."""
        return len(self.history)


print("✅ ChatBot class created!")


✅ ChatBot class created!


In [212]:
# Test our chatbot with memory!
bot = ChatBot(system_prompt="You are a friendly coding tutor. Keep answers short and clear.")

# Turn 1
print("👤 User: My name is Bob and I'm learning Python.")
print(f"🤖 Bot: {bot.chat('My name is Bob and I am learning Python.')}")
print(f"   📊 History size: {bot.get_history_length()} messages\n")

# Turn 2 — Does it remember?
print("👤 User: What's my name and what am I learning?")
print(f"🤖 Bot: {bot.chat('What is my name and what am I learning?')}")
print(f"   📊 History size: {bot.get_history_length()} messages\n")

# Turn 3 — Continuing the conversation
print("👤 User: Give me a tip about what I'm learning.")
print(f"🤖 Bot: {bot.chat('Give me a tip about what I am learning.')}")
print(f"   📊 History size: {bot.get_history_length()} messages")


👤 User: My name is Bob and I'm learning Python.
🤖 Bot: Hello Bob! I'm here to help you learn Python. Let's start with the basics. What would you like to learn first?
   📊 History size: 3 messages

👤 User: What's my name and what am I learning?
🤖 Bot: Your name is Bob and you're learning Python.
   📊 History size: 5 messages

👤 User: Give me a tip about what I'm learning.
🤖 Bot: Tip: In Python, indentation matters! Use 4 spaces or a tab to indent your code blocks.
   📊 History size: 7 messages


### ⚠️ The Sliding Window Problem

As conversations grow, you'll hit the model's **context window limit**. A common solution is a **sliding window** — keep only the last N messages:

```
Full History:     [msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8]
Window (last 4):  [                          msg5, msg6, msg7, msg8]
```

In [213]:
class SlidingWindowChatBot:
    """ChatBot with a sliding window memory to avoid exceeding context limits."""

    def __init__(self, system_prompt: str = "You are a helpful assistant.", max_messages: int = 10) -> None:
        """Initialize the chatbot with a system prompt, empty history, and max message limit."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.system_message = SystemMessage(content=system_prompt)
        self.history: list[BaseMessage] = []
        self.max_messages = max_messages  # Max user+AI messages to keep

    def chat(self, user_message: str) -> str:
        """Send a message and get a response, maintaining a sliding window of history."""
        self.history.append(HumanMessage(content=user_message))

        # Apply sliding window — always keep system message + last N messages
        windowed_history = [self.system_message, *self.history[-self.max_messages:]]

        response = self.model.invoke(windowed_history)
        self.history.append(AIMessage(content=response.content))

        return response.content


# Test with a small window
bot = SlidingWindowChatBot(max_messages=4)

messages = [
    "My favorite color is blue.",
    "I live in Paris.",
    "I work as a data scientist.",
    "What's my favorite color?",  # This might be forgotten with window=4!
]

for msg in messages:
    print(f"👤 {msg}")
    print(f"🤖 {bot.chat(msg)}\n")


👤 My favorite color is blue.
🤖 That's great! Blue is a wonderful color. It's often associated with calmness, tranquility, and stability. Do you have a specific shade of blue that you like the most, such as sky blue, navy blue, or perhaps something else?

👤 I live in Paris.
🤖 Paris is a beautiful city! With its rich history, stunning architecture, and vibrant culture, there's always something to see and do. Since blue is your favorite color, you might enjoy visiting places like the Musée d'Orsay, which has a beautiful blue dome, or taking a stroll along the Seine River, which is often reflected in shades of blue. Have you been to any of these places, or do you have a favorite spot in Paris?

👤 I work as a data scientist.
🤖 That's fascinating! Data science is a field that combines statistics, computer science, and domain expertise to extract insights from structured and unstructured data. As a data scientist in Paris, you're likely surrounded by a thriving tech community and have access 

---

# 🔧 Part 2: Tool Calling - Extending LLM Capabilities

Agents need to interact with the real world, and **tools** are the foundation that allows LLMs to take actions beyond generating text.

```
Without Tools:                  With Tools:
┌──────────────────┐            ┌──────────────────┐
│ User: What's     │            │ User: What's     │
│ 847 × 293?       │            │ 847 × 293?       │
│                  │            │                  │
│ LLM: Hmm... let  │            │ LLM: I'll use    │
│ me try... 248071? │ ← Wrong!  │ the calculator   │
└──────────────────┘            │ → calc(847*293)  │
                                │ → 248,171 ✅     │
                                └──────────────────┘
```

### 🎯 How Tool Calling Works:

1. You **define tools** (functions with descriptions)
2. You send the tool definitions to the LLM along with the user's query
3. The LLM **decides** which tool(s) to use and with what arguments
4. You **execute** the tool and send results back to the LLM
5. The LLM generates a **final response** using the tool results

### 📐 Defining Tools with LangChain

LangChain makes it easy to define tools using the `@tool` decorator. The docstring becomes the tool description that the LLM reads!

In [214]:
from langchain_core.tools import tool


@tool
def calculator(operation: str, x: float, y: float) -> str:
    """Perform a mathematical calculation.

    Args:
        operation: The operation to perform. One of: add, subtract, multiply, divide
        x: The first number
        y: The second number

    """
    operations = {
        "add": x + y,
        "subtract": x - y,
        "multiply": x * y,
        "divide": x / y if y != 0 else "Error: Division by zero",
    }
    result = operations.get(operation, f"Unknown operation: {operation}")
    return f"{x} {operation} {y} = {result}"


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.

    Args:
        city: The name of the city to get weather for

    """
    # Simulated weather data (in production, you'd call a real API)
    weather_data = {
        "paris": "☀️ 22°C, Sunny with light clouds",
        "london": "🌧️ 15°C, Rainy",
        "new york": "⛅ 28°C, Partly cloudy",
        "tokyo": "🌤️ 25°C, Clear skies",
        "san francisco": "🌫️ 18°C, Foggy",
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"Weather in {city}: {weather_data[city_lower]}"
    return f"Weather in {city}: 🌡️ 20°C, Fair weather (simulated)"


print("✅ Tools defined!")
print(f"\n📐 Calculator tool: {calculator.name}")
print(f"   Description: {calculator.description}")
print(f"\n🌤️ Weather tool: {get_weather.name}")
print(f"   Description: {get_weather.description}")


✅ Tools defined!

📐 Calculator tool: calculator
   Description: Perform a mathematical calculation.

Args:
    operation: The operation to perform. One of: add, subtract, multiply, divide
    x: The first number
    y: The second number

🌤️ Weather tool: get_weather
   Description: Get the current weather for a given city.

Args:
    city: The name of the city to get weather for


### 🔗 Binding Tools to the Model

We tell the model which tools are available using `bind_tools()`:

In [215]:
# Bind tools to the model
tools = [calculator, get_weather]
model_with_tools = model.bind_tools(tools)

# Now when we invoke the model, it can CHOOSE to use tools!
response = model_with_tools.invoke("What is 847 multiplied by 293?")

print("📩 Response content:", response.content)
print("\n🔧 Tool calls:", response.tool_calls)
print("\n💡 The model didn't answer directly — it wants to use the calculator tool!")


📩 Response content: 

🔧 Tool calls: [{'name': 'calculator', 'args': {'operation': 'multiply', 'x': 847, 'y': 293}, 'id': '7xFRkWGpc', 'type': 'tool_call'}]

💡 The model didn't answer directly — it wants to use the calculator tool!


### 🔍 Understanding Tool Call Response

When the model wants to use a tool, it returns a `tool_calls` list instead of a direct answer:

```python
response.tool_calls = [
    {
        'name': 'calculator',           # Which tool to call
        'args': {                        # With what arguments
            'operation': 'multiply',
            'x': 847,
            'y': 293
        },
        'id': 'call_abc123'             # Unique ID for this call
    }
]
```

The LLM **doesn't execute the tool** — it just tells you which tool to call and with what arguments. **You** execute the tool and send results back.

In [216]:
# Let's manually execute the tool and complete the conversation
from langchain_core.messages import ToolMessage

# Step 1: Ask the model
user_msg = HumanMessage(content="What is 847 multiplied by 293?")
ai_response = model_with_tools.invoke([user_msg])

print("Step 1 — LLM decides to use a tool:")
print(f"   Tool: {ai_response.tool_calls[0]['name']}")
print(f"   Args: {ai_response.tool_calls[0]['args']}")

# Step 2: Execute the tool
tool_call = ai_response.tool_calls[0]
tool_result = calculator.invoke(tool_call["args"])

print("\nStep 2 — We execute the tool:")
print(f"   Result: {tool_result}")

# Step 3: Send the tool result back to the LLM
tool_msg = ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])

final_response = model_with_tools.invoke([user_msg, ai_response, tool_msg])

print("\nStep 3 — LLM generates final answer:")
print(f"   💬 {final_response.content}")


Step 1 — LLM decides to use a tool:
   Tool: calculator
   Args: {'operation': 'multiply', 'x': 847, 'y': 293}

Step 2 — We execute the tool:
   Result: 847.0 multiply 293.0 = 248171.0

Step 3 — LLM generates final answer:
   💬 The result of 847 multiplied by 293 is 248,171.


### 🌤️ Testing with the Weather Tool

The model **chooses** the right tool based on the question:

In [217]:
# The model will pick the weather tool this time!
user_msg = HumanMessage(content="What's the weather like in Paris?")
ai_response = model_with_tools.invoke([user_msg])

print(f"🔧 Tool chosen: {ai_response.tool_calls[0]['name']}")
print(f"   Args: {ai_response.tool_calls[0]['args']}")

# Execute the tool
tool_call = ai_response.tool_calls[0]
tool_result = get_weather.invoke(tool_call["args"])
print(f"\n🌤️ Tool result: {tool_result}")

# Get final response
tool_msg = ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
final_response = model_with_tools.invoke([user_msg, ai_response, tool_msg])
print(f"\n💬 Final answer: {final_response.content}")


🔧 Tool chosen: get_weather
   Args: {'city': 'Paris'}

🌤️ Tool result: Weather in Paris: ☀️ 22°C, Sunny with light clouds

💬 Final answer: The weather in Paris is currently sunny with light clouds and a temperature of 22°C.


### 💡 When the Model Doesn't Need Tools

If the question doesn't require a tool, the model answers directly:

In [218]:
# A question that doesn't need any tool
response = model_with_tools.invoke("What is the capital of France?")

print(f"🔧 Tool calls: {response.tool_calls}")
print(f"💬 Direct answer: {response.content}")
print("\n✅ The model is smart enough to know it doesn't need a tool here!")


🔧 Tool calls: []
💬 Direct answer: The capital of France is Paris.

✅ The model is smart enough to know it doesn't need a tool here!


---

# 🔄 Part 3: The ReAct Pattern - Reason + Act

The **ReAct** (Reasoning + Acting) pattern is the most common way to build agents. The idea is simple:

```
┌─────────────────────────────────────────┐
│              ReAct Loop                  │
│                                         │
│  1. 🧠 REASON: Analyze the question     │
│           ↓                              │
│  2. 🔧 ACT: Choose & call a tool        │
│           ↓                              │
│  3. 👀 OBSERVE: Get the tool result      │
│           ↓                              │
│  4. 🔄 REPEAT or ✅ ANSWER              │
│                                         │
└─────────────────────────────────────────┘
```

The agent keeps looping until it has enough information to give a final answer.

### 🎯 Key Insight
This is essentially what we did manually in the tool calling section — but automated in a **while loop**!

### 🏗️ Building a ReAct Agent from Scratch

Let's build our own ReAct agent in vanilla Python!
This is the same pattern used by LangGraph, CrewAI, and other frameworks under the hood.

In [219]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import BaseTool


class ReActAgent:
    """A ReAct agent built from scratch.

    Implements the Reason-Act-Observe loop with tool calling.
    """

    def __init__(
        self,
        tools: list[BaseTool],
        system_prompt: str = "You are a helpful assistant with access to tools. Use them when needed.",
        max_iterations: int = 5,
    ) -> None:
        """Initialize the agent with a model, tools, system prompt, and max iterations."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.tools = {t.name: t for t in tools}  # Map tool name → tool object
        self.model_with_tools = self.model.bind_tools(tools)
        self.system_prompt = system_prompt
        self.max_iterations = max_iterations
        self.conversation: list[BaseMessage] = []

    def _execute_tools(self, ai_message: AIMessage) -> list[ToolMessage]:
        """Execute all tool calls from the AI message."""
        tool_messages = []
        for tool_call in ai_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            print(f"   🔧 Calling tool: {tool_name}({tool_args})")

            # Execute the tool
            tool = self.tools[tool_name]
            result = tool.invoke(tool_args)

            print(f"   📋 Result: {result}")

            tool_messages.append(
                ToolMessage(content=str(result), tool_call_id=tool_call["id"]),
            )
        return tool_messages

    def invoke(self, user_message: str) -> str:
        """Run the ReAct loop until the agent produces a final answer."""
        # Initialize conversation
        self.conversation = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=user_message),
        ]

        print(f"👤 User: {user_message}\n")

        for i in range(self.max_iterations):
            print(f"🔄 Iteration {i + 1}:")

            # REASON: Call the LLM
            response = self.model_with_tools.invoke(self.conversation)
            self.conversation.append(response)

            # Check if the model wants to use tools
            if response.tool_calls:
                # ACT: Execute tools
                tool_messages = self._execute_tools(response)

                # OBSERVE: Add tool results to conversation
                self.conversation.extend(tool_messages)
                print()
                continue  # Loop back for more reasoning

            # No tool calls → final answer!
            print(f"\n✅ Final answer: {response.content}")
            return response.content

        return "⚠️ Max iterations reached without a final answer."


print("✅ ReActAgent class created!")


✅ ReActAgent class created!


In [220]:
# Create our agent with both tools
agent = ReActAgent(tools=[calculator, get_weather])

# Test 1: Math question → should use calculator
result = agent.invoke("What is 1234 multiplied by 5678?")


👤 User: What is 1234 multiplied by 5678?

🔄 Iteration 1:
   🔧 Calling tool: calculator({'operation': 'multiply', 'x': 1234, 'y': 5678})
   📋 Result: 1234.0 multiply 5678.0 = 7006652.0

🔄 Iteration 2:

✅ Final answer: The result of 1234 multiplied by 5678 is 7,006,652.


In [221]:
# Test 2: Weather question → should use get_weather
result = agent.invoke("What's the weather in Tokyo?")


👤 User: What's the weather in Tokyo?

🔄 Iteration 1:
   🔧 Calling tool: get_weather({'city': 'Tokyo'})
   📋 Result: Weather in Tokyo: 🌤️ 25°C, Clear skies

🔄 Iteration 2:

✅ Final answer: The weather in Tokyo is currently clear with a temperature of 25°C.


In [222]:
# Test 3: No tool needed
result = agent.invoke("Tell me a fun fact about cats.")


👤 User: Tell me a fun fact about cats.

🔄 Iteration 1:

✅ Final answer: Did you know that cats can make over 100 different sounds, while dogs can only make around 10? This is one of the many fascinating facts about our feline friends.


In [223]:
# Test 4: Multi-step reasoning — requires multiple tool calls
result = agent.invoke(
    "What's the weather in Paris and London? Also, what is 42 divided by 7?",
)


👤 User: What's the weather in Paris and London? Also, what is 42 divided by 7?

🔄 Iteration 1:
   🔧 Calling tool: get_weather({'city': 'Paris'})
   📋 Result: Weather in Paris: ☀️ 22°C, Sunny with light clouds
   🔧 Calling tool: get_weather({'city': 'London'})
   📋 Result: Weather in London: 🌧️ 15°C, Rainy
   🔧 Calling tool: calculator({'operation': 'divide', 'x': 42, 'y': 7})
   📋 Result: 42.0 divide 7.0 = 6.0

🔄 Iteration 2:

✅ Final answer: The weather in Paris is ☀️ 22°C, Sunny with light clouds.

The weather in London is 🌧️ 15°C, Rainy.

42 divided by 7 is 6.


### 🎉 What Did We Just Build?

We built a **ReAct agent** from scratch! This is the same pattern that powers:
- LangGraph's `create_react_agent()`
- OpenAI's function calling agents
- Most modern agent frameworks

The core idea is just a **while loop** + **tool calling**!

---

# 🏗️ Part 4: Using LangGraph for Agents

While building from scratch is educational, frameworks like **LangGraph** make it much easier to build production-ready agents.

LangGraph provides:

| Feature | Benefit |
|---------|--------|
| `create_react_agent()` | One-line agent creation |
| State management | Built-in conversation memory |
| Graph-based flow | Visual, debuggable workflows |
| Streaming | Real-time response streaming |

Let's recreate our agent in just a few lines!

In [224]:
from langchain.agents import create_agent

# Create a ReAct agent in ONE line!
langgraph_agent = create_agent(
    model=ChatMistralAI(model="mistral-small-latest", temperature=0),
    tools=[calculator, get_weather],
)

print("✅ LangGraph agent created in one line!")


✅ LangGraph agent created in one line!


In [225]:
# Invoke the LangGraph agent
response = langgraph_agent.invoke(
    {"messages": [("user", "What is 99 multiplied by 88?")]},
)

# The response contains the full message history
print("📜 Full conversation trace:\n")
for msg in response["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"🤖 {role}: [tool call → {msg.tool_calls[0]['name']}({msg.tool_calls[0]['args']})]")
    elif hasattr(msg, "content") and msg.content:
        print(f"{'👤' if role == 'HumanMessage' else '🤖'} {role}: {msg.content[:200]}")
    else:
        print(f"📋 {role}: {msg}")


📜 Full conversation trace:

👤 HumanMessage: What is 99 multiplied by 88?
🤖 AIMessage: [tool call → calculator({'operation': 'multiply', 'x': 99, 'y': 88})]
🤖 ToolMessage: 99.0 multiply 88.0 = 8712.0
🤖 AIMessage: The result of 99 multiplied by 88 is 8712.


In [226]:
# Test with weather question
response = langgraph_agent.invoke(
    {"messages": [("user", "What's the weather in Paris and New York?")]},
)

# Print just the final answer
final_message = response["messages"][-1]
print(f"💬 Final answer: {final_message.content}")


💬 Final answer: The weather in Paris is ☀️ 22°C, Sunny with light clouds.

The weather in New York is ⛅ 28°C, Partly cloudy.


### 🔍 LangGraph with Streaming

One of the big advantages of LangGraph is built-in streaming support. You can see the agent's reasoning in real time:

In [227]:
# Stream the agent's execution step by step
print("🌊 Streaming agent execution:\n")

for step in langgraph_agent.stream(
    {"messages": [("user", "What is 15 times 23, and what's the weather in London?")]},
):
    # Each step is a dict with the node name as key
    for node_name, node_output in step.items():
        print(f"📍 Node: {node_name}")
        for msg in node_output.get("messages", []):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 Tool call: {tc['name']}({tc['args']})")
            elif hasattr(msg, "content") and msg.content:
                content_preview = msg.content[:150]
                print(f"   💬 {content_preview}")
        print()


🌊 Streaming agent execution:

📍 Node: model
   🔧 Tool call: calculator({'operation': 'multiply', 'x': 15, 'y': 23})
   🔧 Tool call: get_weather({'city': 'London'})

📍 Node: tools
   💬 Weather in London: 🌧️ 15°C, Rainy

📍 Node: tools
   💬 15.0 multiply 23.0 = 345.0

📍 Node: model
   💬 The result of 15 times 23 is 345. The weather in London is currently rainy with a temperature of 15°C.



---

# 🧩 Part 5: Building a Multi-Tool Agent

Let's build a more capable agent with multiple specialized tools. This is closer to what you'd build in production!

We'll add:
- 🔍 A **search** tool (simulated)
- 📝 A **note-taking** tool
- 🕐 A **time** tool

In [228]:
from datetime import UTC, datetime

notes_store: dict[str, str] = {}


@tool
def search_knowledge_base(query: str) -> str:
    """Search a knowledge base for information about programming and technology.

    Args:
        query: The search query

    """
    knowledge = {
        "python": "Python is a high-level programming language created by Guido van Rossum in 1991. It emphasizes code readability and supports multiple paradigms.",
        "langchain": "LangChain is a framework for developing applications powered by language models. It provides tools for chains, agents, and retrieval.",
        "mistral": "Mistral AI is a French AI company founded in 2023. They develop open-weight LLMs including Mistral 7B, Mixtral, and Mistral Large.",
        "rag": "RAG (Retrieval Augmented Generation) is a technique that enhances LLMs by retrieving relevant documents before generating answers.",
        "transformer": "Transformers are neural network architectures introduced in the 'Attention is All You Need' paper (2017). They use self-attention mechanisms.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"No specific results found for '{query}'. Try searching for: python, langchain, mistral, rag, or transformer."


@tool
def save_note(title: str, content: str) -> str:
    """Save a note with a title and content for later retrieval.

    Args:
        title: The title of the note
        content: The content of the note

    """
    notes_store[title] = content
    return f"✅ Note '{title}' saved successfully!"


@tool
def get_notes() -> str:
    """Retrieve all saved notes."""
    if not notes_store:
        return "📭 No notes saved yet."
    result = "📝 Saved notes:\n"
    for title, content in notes_store.items():
        result += f"  - {title}: {content}\n"
    return result


@tool
def get_current_time() -> str:
    """Get the current date and time."""
    now = datetime.now(UTC)
    return f"🕐 Current UTC time: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"


print("✅ All tools defined!")
print("   📐 calculator")
print("   🌤️ get_weather")
print("   🔍 search_knowledge_base")
print("   📝 save_note")
print("   📋 get_notes")
print("   🕐 get_current_time")


✅ All tools defined!
   📐 calculator
   🌤️ get_weather
   🔍 search_knowledge_base
   📝 save_note
   📋 get_notes
   🕐 get_current_time


In [229]:
# Create a powerful multi-tool agent
all_tools = [calculator, get_weather, search_knowledge_base, save_note, get_notes, get_current_time]

multi_agent = create_agent(
    model=ChatMistralAI(model="mistral-small-latest", temperature=0),
    tools=all_tools,
    system_prompt="You are a helpful research assistant. You have access to multiple tools including a calculator, weather service, knowledge base search, note-taking, and a clock. Use the appropriate tools to help the user. Always be thorough and use multiple tools if needed.",
)

print("✅ Multi-tool agent ready with 6 tools!")


✅ Multi-tool agent ready with 6 tools!


In [230]:
# Test: Complex multi-tool query
response = multi_agent.invoke(
    {"messages": [("user", "Search for information about Mistral AI and save it as a note titled 'Mistral Info'. Then tell me the current time.")]},
)

# Print the conversation trace
print("📜 Agent trace:\n")
for msg in response["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"   🔧 Tool: {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"   📋 Result: {msg.content[:100]}...")
    elif role == "HumanMessage":
        print(f"   👤 User: {msg.content}")
    elif msg.content:
        print(f"\n💬 Final: {msg.content}")


📜 Agent trace:

   👤 User: Search for information about Mistral AI and save it as a note titled 'Mistral Info'. Then tell me the current time.
   🔧 Tool: search_knowledge_base({'query': 'Mistral AI'})
   🔧 Tool: get_current_time({})
   📋 Result: Mistral AI is a French AI company founded in 2023. They develop open-weight LLMs including Mistral 7...
   📋 Result: 🕐 Current UTC time: 2026-03-04 17:18:07 UTC...
   🔧 Tool: save_note({'title': 'Mistral Info', 'content': 'Mistral AI is a French AI company founded in 2023. They develop open-weight LLMs including Mistral 7B, Mixtral, and Mistral Large.'})
   📋 Result: ✅ Note 'Mistral Info' saved successfully!...

💬 Final: I have saved the information about Mistral AI as a note titled 'Mistral Info'. The current time is 2026-03-04 17:18:07 UTC.


In [231]:
# Verify the note was saved
print("📝 Notes in store:")
for title, content in notes_store.items():
    print(f"   {title}: {content}")


📝 Notes in store:
   Mistral Info: Mistral AI is a French AI company founded in 2023. They develop open-weight LLMs including Mistral 7B, Mixtral, and Mistral Large.


---

# 🎯 Part 6: Structured Output with Tool Calling

A powerful pattern is using tool calling to get **structured output** from the LLM. Instead of parsing free-text, we force the model to return data in a specific Pydantic schema.

```
Unstructured:                  Structured:
"The sentiment is positive     {"sentiment": "positive",
 with confidence around 90%"    "confidence": 0.9,
                                "keywords": ["great", "love"]}
```

In [232]:
from pydantic import BaseModel, Field


class SentimentAnalysis(BaseModel):
    """Analyze the sentiment of a piece of text."""

    sentiment: str = Field(description="The sentiment: positive, negative, or neutral")
    confidence: float = Field(description="Confidence score between 0 and 1")
    keywords: list[str] = Field(description="Key words that influenced the sentiment")
    summary: str = Field(description="One-sentence summary of the analysis")


def to_sentiment_result(raw: object) -> SentimentAnalysis:
    """Normalize structured output into a SentimentAnalysis model."""
    if isinstance(raw, SentimentAnalysis):
        return raw
    if isinstance(raw, BaseModel):
        return SentimentAnalysis.model_validate(raw.model_dump())
    if isinstance(raw, dict):
        return SentimentAnalysis.model_validate(raw)
    return SentimentAnalysis.model_validate_json(str(raw))


structured_model = model.with_structured_output(SentimentAnalysis)
raw_result = structured_model.invoke(
    "I absolutely love this new Python library! It makes everything so much easier and the documentation is fantastic.",
)
result = to_sentiment_result(raw_result)

print("📊 Sentiment Analysis Result:")
print(f"   Sentiment:  {result.sentiment}")
print(f"   Confidence: {result.confidence}")
print(f"   Keywords:   {result.keywords}")
print(f"   Summary:    {result.summary}")
print(f"\n   Type: {type(result).__name__} ← It's a proper Pydantic object!")


📊 Sentiment Analysis Result:
   Sentiment:  positive
   Confidence: 0.95
   Keywords:   ['love', 'easier', 'fantastic']
   Summary:    The text expresses strong positive sentiment about a new Python library, highlighting ease of use and excellent documentation.

   Type: SentimentAnalysis ← It's a proper Pydantic object!


In [233]:
class PersonInfo(BaseModel):
    """Extract personal information from text."""

    name: str = Field(description="The person's full name")
    age: int | None = Field(description="The person's age if mentioned")
    occupation: str | None = Field(description="The person's job or occupation if mentioned")
    location: str | None = Field(description="Where the person lives if mentioned")
    interests: list[str] = Field(description="The person's interests or hobbies")


def to_person_info(raw: object) -> PersonInfo:
    """Normalize structured output into a PersonInfo model."""
    if isinstance(raw, PersonInfo):
        return raw
    if isinstance(raw, BaseModel):
        return PersonInfo.model_validate(raw.model_dump())
    if isinstance(raw, dict):
        return PersonInfo.model_validate(raw)
    return PersonInfo.model_validate_json(str(raw))


person_extractor = model.with_structured_output(PersonInfo)
raw_result = person_extractor.invoke(
    "Marie is a 28-year-old machine learning engineer living in Lyon. "
    "She enjoys hiking, reading science fiction novels, and contributing to open source projects.",
)
result = to_person_info(raw_result)

print("👤 Extracted Person Info:")
print(f"   Name:        {result.name}")
print(f"   Age:         {result.age}")
print(f"   Occupation:  {result.occupation}")
print(f"   Location:    {result.location}")
print(f"   Interests:   {result.interests}")


👤 Extracted Person Info:
   Name:        Marie
   Age:         28
   Occupation:  Machine Learning Engineer
   Location:    Lyon
   Interests:   ['hiking', 'reading science fiction novels', 'contributing to open source projects']


---

# 🧠 Part 7: Putting It All Together — A Complete Agent

Let's build a complete agent that combines **memory**, **tools**, and **structured output**. This agent acts as a personal research assistant.

```
┌─────────────────────────────────────────────────────┐
│           Complete Research Agent                     │
│                                                       │
│  📝 System Prompt (personality & rules)               │
│  🧠 Memory (conversation history)                     │
│  🔧 Tools:                                           │
│     ├── 📐 Calculator                                │
│     ├── 🌤️ Weather                                   │
│     ├── 🔍 Knowledge Base                            │
│     ├── 📝 Note Taking                               │
│     └── 🕐 Clock                                     │
│  🔄 ReAct Loop (reason → act → observe → repeat)     │
│                                                       │
└─────────────────────────────────────────────────────┘
```

In [234]:

from langchain_core.messages import HumanMessage


class ResearchAssistant:
    """A complete research assistant agent with memory, tools, and conversation tracking."""

    def __init__(self) -> None:
        """Initialize the research assistant with a model, tools, system prompt, and empty conversation history."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.tools = [calculator, get_weather, search_knowledge_base, save_note, get_notes, get_current_time]

        system_prompt = (
            "You are a research assistant powered by Mistral AI. "
            "You can search for information, do calculations, check the weather, "
            "and take notes. Always be thorough — use tools when they can help. "
            "When you search for information, save important findings as notes for future reference."
        )

        self.agent = create_agent(
            model=self.model,
            tools=self.tools,
            system_prompt=system_prompt,
        )

        # Conversation memory
        self.messages: list = []

    def chat(self, user_message: str) -> str:
        """Send a message to the agent and get a response."""
        self.messages.append(("user", user_message))

        response = self.agent.invoke({"messages": self.messages})

        # Extract the final AI message
        final_msg = response["messages"][-1].content
        self.messages.append(("assistant", final_msg))

        return final_msg

    def get_conversation_length(self) -> int:
        """Get the number of messages in the conversation history."""
        return len(self.messages)


print("✅ ResearchAssistant ready!")


✅ ResearchAssistant ready!


In [236]:
# Create the assistant
assistant = ResearchAssistant()

# Multi-turn conversation with the agent
queries = [
    "Hello! Can you search for information about RAG and save it as a note?",
    "Great! Now what's 256 divided by 16?",
    "What's the weather in Paris right now?",
    "Can you show me all the notes we've saved so far?",
]

for query in queries:
    print(f"\n{'='*70}")
    print(f"👤 User: {query}")
    response = assistant.chat(query)
    print(f"🤖 Agent: {response}")

print(f"\n📊 Conversation length: {assistant.get_conversation_length()} messages")



👤 User: Hello! Can you search for information about RAG and save it as a note?
🤖 Agent: I've saved a note about RAG.

👤 User: Great! Now what's 256 divided by 16?
🤖 Agent: The result of 256 divided by 16 is 16.

👤 User: What's the weather in Paris right now?
🤖 Agent: The current weather in Paris is sunny with light clouds and a temperature of 22°C.

👤 User: Can you show me all the notes we've saved so far?
🤖 Agent: Here are the notes we've saved so far:

1. Title: RAG
Content: RAG (Red-Ambler-Green) is a project management and performance reporting tool that provides a visual indication of the status of tasks or projects using three colors: Red (R), Amber (A), and Green (G). It helps teams and managers quickly understand the current state of various projects or tasks. Red indicates a critical issue or significant delay, Amber signifies a warning or slight delay, and Green shows that everything is on track and progressing as planned. RAG status is often used in project management, risk

---

## 📝 Summary

Congratulations! 🎉 You've learned to build AI agents from scratch!

✅ **Memory** — Maintaining conversation history across turns  
✅ **Tool Calling** — Letting LLMs use functions to take actions  
✅ **ReAct Pattern** — The reason-act-observe loop  
✅ **LangGraph** — Building agents with frameworks  
✅ **Multi-Tool Agents** — Combining multiple capabilities  
✅ **Structured Output** — Getting typed data from LLMs  

### 🔑 Key Takeaways:

| Concept | One-liner |
|---------|----------|
| **Agent** | An LLM that can reason AND act |
| **Memory** | Pass conversation history on each call |
| **Tools** | Functions the LLM can decide to call |
| **ReAct** | While loop: reason → use tool → observe → repeat |
| **LangGraph** | Framework that handles the loop for you |
| **Structured Output** | Force LLM to return Pydantic models |

### 🚀 Next Steps:

- Add **real API tools** (web search, database queries)
- Explore **multi-agent systems** (agents that coordinate with each other)
- Learn about **Model Context Protocol (MCP)** for standardized tool interfaces
- Build **RAG agents** that retrieve from real vector databases

### 📚 Resources:

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Mistral AI Function Calling](https://docs.mistral.ai/capabilities/function_calling/)
- [LangChain Agents](https://python.langchain.com/docs/concepts/agents/)
- [ReAct Paper](https://arxiv.org/abs/2210.03629)

---

**Happy agent building!** 🚀✨

---

## 🏋️ Exercises

Time to practice! The exercises below go from fundamental memory concepts to advanced agent patterns. Each exercise builds on what you learned in this notebook.

### 📚 Exercise 1: Summarized Memory ChatBot

Instead of a sliding window that **drops** old messages, implement a chatbot that **summarizes** them. When the history exceeds `max_messages`, use the LLM itself to create a summary, then continue the conversation with the summary as context.

**Requirements:**
- Implement `_summarize_history()` using the LLM to condense old messages
- Replace old messages with a `SystemMessage` containing the summary
- Keep the most recent messages intact
- The bot should still remember key facts after summarization

In [237]:
class SummarizedMemoryChatBot:
    """A chatbot that summarizes old messages instead of dropping them.

    When the history exceeds max_messages, it:
    1. Sends old messages to the LLM with a summarization prompt
    2. Replaces them with a summary context
    3. Keeps recent messages intact
    """

    def __init__(self, system_prompt: str = "You are a helpful assistant.", max_messages: int = 6) -> None:
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.system_prompt = system_prompt
        self.history: list[BaseMessage] = []
        self.max_messages = max_messages
        self.summary: str = ""

    @staticmethod
    def _to_text(content: str | list[str | dict]) -> str:
        if isinstance(content, str):
            return content
        return str(content)

    def _summarize_history(self) -> str:
        """Summarize older conversation turns into a concise memory."""
        if not self.history:
            return ""

        conversation_text = "\n".join(
            f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {self._to_text(m.content)}"
            for m in self.history
            if isinstance(m, (HumanMessage, AIMessage))
        )

        summary_messages = [
            SystemMessage(
                content=(
                    "You summarize conversations. Keep only key facts, preferences, goals, "
                    "and commitments that the assistant should remember."
                ),
            ),
            HumanMessage(content=conversation_text),
        ]
        summary_response = self.model.invoke(summary_messages)
        return self._to_text(summary_response.content)

    def chat(self, user_message: str) -> str:
        """Chat with memory. When history gets too long, summarize older messages."""
        if len(self.history) >= self.max_messages:
            self.summary = self._summarize_history()
            self.history = []

        full_messages: list[BaseMessage] = [SystemMessage(content=self.system_prompt)]
        if self.summary:
            full_messages.append(
                SystemMessage(
                    content=f"Summary of previous conversation:\n{self.summary}",
                ),
            )
        full_messages.extend(self.history)
        full_messages.append(HumanMessage(content=user_message))

        response = self.model.invoke(full_messages)
        response_text = self._to_text(response.content)

        self.history.append(HumanMessage(content=user_message))
        self.history.append(AIMessage(content=response_text))
        return response_text


bot = SummarizedMemoryChatBot(max_messages=4)

test_messages = [
    "My name is Alice and I love machine learning.",
    "I work at Google as a research scientist.",
    "I recently published a paper on transformers.",
    "I also enjoy hiking on weekends.",
    "What do you know about me so far?",
    "What was my paper about?",
]

for msg in test_messages:
    print(f"\n👤 {msg}")
    response = bot.chat(msg)
    print(f"🤖 {response}")
    print(f"   📊 History: {len(bot.history)} messages | Summary: {'Yes' if bot.summary else 'No'}")



👤 My name is Alice and I love machine learning.
🤖 Hello Alice! It's great to meet you. With your interest in machine learning, I'm sure you're always looking for new ways to learn and apply your skills. How can I assist you today? Are you looking for resources, project ideas, or help with a specific concept in machine learning?
   📊 History: 2 messages | Summary: No

👤 I work at Google as a research scientist.
🤖 That's impressive, Alice! Working as a research scientist at Google in the field of machine learning must be incredibly exciting and rewarding. With your expertise, I'm sure you're involved in cutting-edge projects and research. How can I assist you today? Are you looking for information on a specific topic, collaboration ideas, or perhaps help with a technical challenge you're facing? I'm here to help!
   📊 History: 4 messages | Summary: No

👤 I recently published a paper on transformers.
🤖 That's great to hear! Congratulations on your publication. Transformers are a fascinat

### 🔧 Exercise 2: Build Your Own Custom Tools

Create 3 custom tools and wire them into a LangGraph agent:

1. **`translate`** — Translates text to a target language (use the LLM itself as the translation engine)
2. **`string_analyzer`** — Returns stats about a string (word count, character count, most common word)
3. **`random_fact`** — Returns a random fun fact from a hardcoded list

Then test your agent with:
- "Translate 'Hello, how are you?' to French"
- "Analyze the string: 'The quick brown fox jumps over the lazy dog'"
- "Tell me a random fact and then translate it to Spanish"

In [241]:
import secrets
from collections import Counter

from langchain_core.tools import tool


def _content_to_text(content: str | list[str | dict]) -> str:
    if isinstance(content, str):
        return content
    return str(content)


@tool
def translate(text: str, target_language: str) -> str:
    """Translate text to a target language.

    Args:
        text: The text to translate
        target_language: The language to translate to (e.g., French, Spanish, German)

    """
    prompt = f"Translate the following text to {target_language}:\n\n{text}"
    response = ChatMistralAI(model="mistral-small-latest", temperature=0).invoke(prompt)
    return _content_to_text(response.content).strip()


@tool
def string_analyzer(text: str) -> str:
    """Analyze a string and return statistics.

    Args:
        text: The text to analyze

    """
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    counts = Counter(words)
    most_common_word, occurrences = counts.most_common(1)[0] if counts else ("", 0)
    return (
        f"String analysis:\n"
        f"  - Word count: {word_count}\n"
        f"  - Character count: {char_count}\n"
        f"  - Unique words: {len(counts)}\n"
        f"  - Most common word: '{most_common_word}' (appears {occurrences} times)"
    )


@tool
def random_fact() -> str:
    """Return a random fun fact."""
    facts = [
        "Honey never spoils. Archaeologists have found 3000-year-old honey in Egyptian tombs that was still edible.",
        "Octopuses have three hearts and blue blood.",
        "A group of flamingos is called a 'flamboyance'.",
        "The shortest war in history lasted 38 minutes between Britain and Zanzibar in 1896.",
        "Bananas are berries, but strawberries are not.",
    ]
    return secrets.choice(facts)


custom_agent = create_agent(
    model=ChatMistralAI(model="mistral-small-latest", temperature=0),
    tools=[translate, string_analyzer, random_fact],
)

test_queries = [
    "Translate 'Hello, how are you?' to French",
    "Analyze the string: 'The quick brown fox jumps over the lazy dog'",
    "Tell me a random fact and then translate it to Spanish",
]

for query in test_queries:
    response = custom_agent.invoke({"messages": [("user", query)]})
    final = _content_to_text(response["messages"][-1].content)
    print(f"\n👤 {query}")
    print(f"🤖 {final}")




👤 Translate 'Hello, how are you?' to French
🤖 Bonjour, comment ça va ?

Alternatively, you could also say:
- Salut, comment ça va ? (more casual)
- Bonjour, comment allez-vous ? (more formal)

👤 Analyze the string: 'The quick brown fox jumps over the lazy dog'
🤖 The analysis of the string "The quick brown fox jumps over the lazy dog" is as follows:

- **Word count:** 9
- **Character count:** 43
- **Unique words:** 9
- **Most common word:** "The" (appears 1 time)

👤 Tell me a random fact and then translate it to Spanish
🤖 Here is a random fact: "Bananas are berries, but strawberries are not."

And here is the translation to Spanish:

"Los plátanos son bayas, pero las fresas no lo son."


### 🔄 Exercise 3: Manual ReAct Loop with Trace Logging

Extend the `ReActAgent` class from Part 3 to produce a detailed **trace log** of every step the agent takes. The trace should record each iteration, the tool chosen, the arguments, the result, and the final answer.

**Requirements:**
- Add a `trace: List[dict]` attribute to the agent
- Each trace entry should have: `iteration`, `action` (tool name or "final_answer"), `input`, `output`
- Add a `print_trace()` method that displays the trace in a formatted table
- Test with a query that requires multiple tool calls

In [242]:
class TracedReActAgent:
    """A ReAct agent that logs a detailed trace of every step."""

    def __init__(
        self,
        tools: list[BaseTool],
        system_prompt: str = "You are a helpful assistant with access to tools.",
        max_iterations: int = 5,
    ) -> None:
        """Initialize the agent with a model, tools, system prompt, max iterations, and empty conversation and trace."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.tools = {t.name: t for t in tools}
        self.model_with_tools = self.model.bind_tools(tools)
        self.system_prompt = system_prompt
        self.max_iterations = max_iterations
        self.conversation: list[BaseMessage] = []
        self.trace: list[dict] = []

    @staticmethod
    def _to_text(content: str | list[str | dict]) -> str:
        if isinstance(content, str):
            return content
        return str(content)

    def _execute_tools(self, ai_message: AIMessage, iteration: int) -> list[ToolMessage]:
        """Execute all tool calls from the AI message and log the trace."""
        tool_messages: list[ToolMessage] = []
        for index, tool_call in enumerate(ai_message.tool_calls):
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool = self.tools[tool_name]
            result = tool.invoke(tool_args)
            result_text = self._to_text(result)

            self.trace.append(
                {
                    "iteration": iteration,
                    "action": tool_name,
                    "input": tool_args,
                    "output": result_text,
                },
            )

            tool_call_id = tool_call.get("id")
            if not isinstance(tool_call_id, str) or not tool_call_id:
                tool_call_id = f"call_{iteration}_{index}"

            tool_messages.append(
                ToolMessage(content=result_text, name=tool_name, tool_call_id=tool_call_id),
            )
        return tool_messages

    def invoke(self, user_message: str) -> str:
        """Run the ReAct loop until the agent produces a final answer, while logging each step in the trace."""
        self.trace = []
        self.conversation = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=user_message),
        ]

        for i in range(self.max_iterations):
            response = self.model_with_tools.invoke(self.conversation)
            self.conversation.append(response)

            if response.tool_calls:
                tool_messages = self._execute_tools(response, iteration=i + 1)
                self.conversation.extend(tool_messages)
                continue

            final_text = self._to_text(response.content)
            self.trace.append(
                {
                    "iteration": i + 1,
                    "action": "final_answer",
                    "input": "-",
                    "output": final_text,
                },
            )
            return final_text

        max_iter_msg = "⚠️ Max iterations reached without a final answer."
        self.trace.append(
            {
                "iteration": self.max_iterations,
                "action": "final_answer",
                "input": "-",
                "output": max_iter_msg,
            },
        )
        return max_iter_msg

    def print_trace(self) -> None:
        """Print the agent's trace in a readable format."""
        print(f"{'Step':<5} | {'Action':<12} | {'Input':<40} | Output")
        print("-" * 120)
        for entry in self.trace:
            step = entry["iteration"]
            action = entry["action"]
            input_str = str(entry["input"])
            output_str = str(entry["output"])
            print(f"{step:<5} | {action:<12} | {input_str[:40]:<40} | {output_str[:80]}")


traced_agent = TracedReActAgent(tools=[calculator, get_weather])
max_retries = 5

for attempt in range(max_retries):
    try:
        result = traced_agent.invoke("What is 15 times 23, and what's the weather in Tokyo?")
        break
    except Exception as exc:
        error_msg = str(exc).lower()
        if "429" in error_msg or "rate limit" in error_msg:
            wait_seconds = min(2**attempt, 30)
            print(f"Rate limit atteint. Nouvelle tentative dans {wait_seconds}s...")
            time.sleep(wait_seconds)
            continue
        raise
else:
    result = "⚠️ Échec après plusieurs tentatives (rate limit)."

print(result)
traced_agent.print_trace()


The result of 15 times 23 is 345.

The weather in Tokyo is currently clear skies with a temperature of 25°C.
Step  | Action       | Input                                    | Output
------------------------------------------------------------------------------------------------------------------------
1     | calculator   | {'operation': 'multiply', 'x': 15, 'y':  | 15.0 multiply 23.0 = 345.0
1     | get_weather  | {'city': 'Tokyo'}                        | Weather in Tokyo: 🌤️ 25°C, Clear skies
2     | final_answer | -                                        | The result of 15 times 23 is 345.

The weather in Tokyo is currently clear skies


### 🎯 Exercise 4: Structured Data Extraction Pipeline

Build an agent that extracts structured information from unstructured text using `with_structured_output()`:

1. Define a `MovieReview` Pydantic model with fields: `title`, `rating` (1-5), `sentiment`, `pros: List[str]`, `cons: List[str]`, `recommended: bool`
2. Create a function that takes a raw text review and returns a `MovieReview`
3. Process a batch of 3 reviews and display results in a formatted table

**Bonus:** Add a `ReviewSummary` model that takes a list of `MovieReview`s and produces an overall recommendation.

In [243]:
from pydantic import BaseModel, Field


# Step 1: Define the MovieReview model
class MovieReview(BaseModel):
    """Extract structured information from a movie review."""

    title: str = Field(description="The movie title")
    rating: int = Field(description="Rating from 1 to 5 stars")
    sentiment: str = Field(description="Overall sentiment: positive, negative, or mixed")
    pros: list[str] = Field(description="List of positive points mentioned")
    cons: list[str] = Field(description="List of negative points mentioned")
    recommended: bool = Field(description="Whether the reviewer recommends the movie")

# Step 2: Create the extraction function
def extract_review(raw_text: str) -> MovieReview:
    """Extract structured data from a raw movie review."""
    structured_model = model.with_structured_output(MovieReview)
    result = structured_model.invoke(raw_text)
    if isinstance(result, MovieReview):
        return result
    return MovieReview.model_validate(result)

# Step 3: Test with sample reviews
raw_reviews = [
    "Inception is a masterpiece! The layers of dreams within dreams blew my mind. "
    "Nolan's direction is impeccable and the cast is stellar. The ending still haunts me. "
    "Only downside is it can be confusing on first watch. 5/5, must see!",

    "I watched The Room last night. Wow, what a disaster. The acting is terrible, "
    "the plot makes no sense, and the dialogue is laughable. But honestly? "
    "It's so bad it's entertaining. I'd give it 2/5.",

    "Interstellar was visually stunning and the science was fascinating. "
    "The relationship between father and daughter was moving. However, "
    "the pacing drags in the middle and some plot points feel forced. "
    "Overall a good watch. 4/5.",
]

for review_text in raw_reviews:
    review = extract_review(review_text)
    print(f"🎬 {review.title} — {'⭐' * review.rating} — {review.sentiment}")
    print(f"   ✅ Pros: {review.pros}")
    print(f"   ❌ Cons: {review.cons}")
    print(f"   👍 Recommended: {review.recommended}\n")

# BONUS: Create a ReviewSummary model
class ReviewSummary(BaseModel):
    """Summarize multiple movie reviews."""

    total_reviews: int
    average_rating: float
    best_movie: str
    overall_recommendation: str



🎬 Inception — ⭐⭐⭐⭐⭐ — positive
   ✅ Pros: ['Layers of dreams within dreams', "Nolan's direction", 'Stellar cast', 'Haunting ending']
   ❌ Cons: ['Can be confusing on first watch']
   👍 Recommended: True

🎬 The Room — ⭐⭐ — mixed
   ✅ Pros: ["It's so bad it's entertaining"]
   ❌ Cons: ['Terrible acting', 'Plot makes no sense', 'Laughable dialogue']
   👍 Recommended: False

🎬 Interstellar — ⭐⭐⭐⭐ — mixed
   ✅ Pros: ['visually stunning', 'fascinating science', 'moving father-daughter relationship']
   ❌ Cons: ['pacing drags in the middle', 'some plot points feel forced']
   👍 Recommended: True



### 🌡️ Exercise 5: Temperature Impact on Agent Behavior

Investigate how `temperature` affects an agent's **tool-calling decisions** and **final answers**.

**Requirements:**
1. Create the same LangGraph agent (with `calculator` and `get_weather`) at 3 different temperatures: `0.0`, `0.5`, `1.0`
2. Send the same ambiguous query to all 3: "I'm going to Paris next week, anything I should know?"
3. Compare: Does the agent decide to call tools? Which ones? How does the answer change?
4. Run each temperature 3 times and track how consistent the tool choices are
5. Print a comparison table

In [244]:
def run_temperature_experiment(query: str, temperatures: list[float], runs_per_temp: int = 3) -> None:
    """Test how temperature affects agent tool-calling decisions.

    For each temperature:
    1. Create a LangGraph agent with that temperature
    2. Send the same query `runs_per_temp` times
    3. Track which tools were called each time
    4. Print a comparison table

    Args:
        query: The user query to test
        temperatures: List of temperature values
        runs_per_temp: Number of times to run each temperature

    """
    for temp in temperatures:
        agent = create_agent(
            model=ChatMistralAI(model="mistral-small-latest", temperature=temp),
            tools=[calculator, get_weather],
        )
        for run in range(1, runs_per_temp + 1):
            response = agent.invoke({"messages": [("user", query)]})
            tool_calls = []
            for msg in response["messages"]:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    tool_calls.extend([tc["name"] for tc in msg.tool_calls])
            answer_preview = response["messages"][-1].content[:50] + "..."
            print(f"Temperature: {temp:<5} | Run: {run} | Tools Used: {tool_calls} | Answer Preview: {answer_preview}")


# Test it
run_temperature_experiment(
    query="I'm going to Paris next week, anything I should know?",
    temperatures=[0.0, 0.5, 1.0],
    runs_per_temp=3,
)



Temperature: 0.0   | Run: 1 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 0.0   | Run: 2 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 0.0   | Run: 3 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 0.5   | Run: 1 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 0.5   | Run: 2 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 0.5   | Run: 3 | Tools Used: [] | Answer Preview: Paris is a beautiful city with a rich history and ...
Temperature: 1.0   | Run: 1 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 1.0   | Run: 2 | Tools Used: [] | Answer Preview: That's exciting! Paris is a beautiful city with a ...
Temperature: 1.0   | Run: 3 | Tools Used: [] | Answer Preview: T

### 🛡️ Exercise 6: Agent with Error Handling & Retries

Real-world tools can fail! Build an agent that **gracefully handles tool errors** and retries.

**Requirements:**
1. Create a `flaky_api` tool that randomly fails 50% of the time (raises an exception)
2. Build a custom ReAct agent (not LangGraph) that:
   - Catches tool execution errors
   - Sends the error message back to the LLM as a `ToolMessage` with the error
   - Lets the LLM decide to retry or use a different approach
   - Has a maximum of 3 retries per tool call
3. Test with a query that triggers the flaky tool

In [245]:


# Step 1: Create a flaky tool
@tool
def flaky_stock_price(ticker: str) -> str:
    """Get the current stock price for a company ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, GOOGL, MSFT)

    """
    # 50% chance of failure
    if secrets.randbelow(2) == 0:
        message = f"API timeout: Could not reach stock service for {ticker}"
        raise ConnectionError(message)

    ticker_upper = ticker.upper()
    prices: dict[str, float] = {
        "AAPL": 187.50,
        "GOOGL": 142.30,
        "MSFT": 415.80,
        "AMZN": 185.60,
    }

    if ticker_upper in prices:
        price = prices[ticker_upper]
    else:
        # pseudo-random fallback price in [50.00, 500.00]
        cents = 5000 + secrets.randbelow(45001)
        price = cents / 100

    return f"{ticker_upper}: ${price:.2f}"


# Step 2: Build a RobustReActAgent
class RobustReActAgent:
    """A ReAct agent that handles tool execution errors gracefully.

    When a tool fails, it sends the error back to the LLM and lets it decide what to do.
    """

    def __init__(self, tools: list[BaseTool], max_iterations: int = 8, max_retries: int = 3) -> None:
        """Initialize the agent with a model, tools, max iterations, max retries, and conversation history."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.tools = {t.name: t for t in tools}
        self.model_with_tools = self.model.bind_tools(tools)
        self.max_iterations = max_iterations
        self.max_retries = max_retries
        self.conversation: list[BaseMessage] = []
        self.retry_counts: dict[str, int] = {}

    @staticmethod
    def _content_to_text(content: str | list[str | dict]) -> str:
        """Normalize model/tool content to plain string."""
        if isinstance(content, str):
            return content
        return str(content)

    def _execute_tools(self, ai_message: AIMessage) -> list[ToolMessage]:
        """Execute tool calls with retry-aware error feedback."""
        tool_messages: list[ToolMessage] = []

        for index, tool_call in enumerate(ai_message.tool_calls):
            tool_name = tool_call.get("name")
            raw_args = tool_call.get("args", {})
            tool_args = raw_args if isinstance(raw_args, dict) else {}

            raw_call_id = tool_call.get("id")
            call_id = raw_call_id if isinstance(raw_call_id, str) and raw_call_id else f"tool_call_{index}"

            if tool_name is None or tool_name not in self.tools:
                tool_messages.append(
                    ToolMessage(
                        content=f"Unknown tool: {tool_name}. Please answer without this tool.",
                        name=tool_name or "unknown_tool",
                        tool_call_id=call_id,
                    ),
                )
                continue

            tool = self.tools[tool_name]

            try:
                result = tool.invoke(tool_args)
                tool_messages.append(
                    ToolMessage(
                        content=self._content_to_text(result),
                        name=tool_name,
                        tool_call_id=call_id,
                    ),
                )
            except (ConnectionError, TimeoutError, ValueError, TypeError, KeyError, RuntimeError) as exc:
                retries = self.retry_counts.get(call_id, 0) + 1
                self.retry_counts[call_id] = retries

                if retries <= self.max_retries:
                    content = f"Error executing {tool_name}: {exc!s}. You can retry this tool."
                else:
                    content = (
                        f"Error executing {tool_name}: {exc!s}. "
                        f"Tool failed after {self.max_retries} retries. Please answer without this tool."
                    )

                tool_messages.append(
                    ToolMessage(
                        content=content,
                        name=tool_name,
                        tool_call_id=call_id,
                    ),
                )

        return tool_messages

    def invoke(self, user_message: str) -> str:
        """Run a ReAct loop with robust tool error handling."""
        self.retry_counts = {}
        self.conversation = [HumanMessage(content=user_message)]

        for _ in range(self.max_iterations):
            ai_message = self.model_with_tools.invoke(self.conversation)
            self.conversation.append(ai_message)

            if not ai_message.tool_calls:
                return self._content_to_text(ai_message.content)

            tool_messages = self._execute_tools(ai_message)
            self.conversation.extend(tool_messages)

        return "⚠️ Max iterations reached without a final answer."


# Test it!
robust_agent = RobustReActAgent(tools=[flaky_stock_price, calculator])
result = robust_agent.invoke("What's the stock price of Apple (AAPL)?")
print(f"\n💬 {result}")



💬 It seems that the stock price service is currently unavailable. Would you like me to try again, or is there something else I can assist you with?


### 📋 Exercise 7: System Prompt Engineering for Agents

System prompts **dramatically** change how an agent behaves — which tools it prefers, how verbose it is, how it reasons.

**Requirements:**
1. Create the same LangGraph agent (with `calculator`, `get_weather`, `search_knowledge_base`) 3 times, each with a different system prompt:
   - **"Concise Agent"**: Always use tools when possible. Give short, direct answers.
   - **"Verbose Agent"**: Explain your reasoning step by step. Always use tools to verify.
   - **"Lazy Agent"**: Only use tools when absolutely necessary. Prefer answering from your own knowledge.
2. Send the same 3 queries to each agent
3. Compare: answer length, number of tool calls, quality of answers
4. Present results in a formatted comparison

In [246]:
system_prompts = {
    "Concise": (
        "You are a concise assistant. Always use your tools when they are relevant. "
        "Give short, direct answers — no more than 2 sentences."
    ),
    "Verbose": (
        "You are a thorough assistant. Explain your reasoning step by step. "
        "Always use tools to verify facts before answering. Provide detailed explanations."
    ),
    "Lazy": (
        "You are an efficient assistant. Only use tools when absolutely necessary — "
        "prefer answering from your own knowledge. Don't call tools for things you already know."
    ),
}

test_queries = [
    "What is 42 times 58?",
    "What's the weather in London?",
    "Tell me about Python programming",
]


def _result_to_text(result: object) -> str:
    if isinstance(result, str):
        return result
    if hasattr(result, "content"):
        content = result.content
        if isinstance(content, str):
            return content
        return str(content)
    return str(result)


def compare_system_prompts(prompts: dict[str, str], queries: list[str]) -> None:
    """Create an agent for each system prompt, run all queries, and compare results."""
    for prompt_name, prompt_text in prompts.items():
        agent = create_agent(
            model=ChatMistralAI(model="mistral-small-latest", temperature=0),
            tools=[calculator, get_weather, search_knowledge_base],
            system_prompt=prompt_text,
        )
        for query in queries:
            response = agent.invoke({"messages": [("user", query)]})
            tool_calls = []
            for msg in response["messages"]:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    tool_calls.extend([tc["name"] for tc in msg.tool_calls])
            final_answer = _result_to_text(response["messages"][-1])
            answer_length = len(final_answer)
            print(
                f"Prompt: {prompt_name:<8} | Query: {query:<25} | "
                f"Tools Used: {tool_calls} | Tool Calls: {len(tool_calls)} | Answer Length: {answer_length}",
            )


compare_system_prompts(system_prompts, test_queries)


Prompt: Concise  | Query: What is 42 times 58?      | Tools Used: ['calculator'] | Tool Calls: 1 | Answer Length: 4
Prompt: Concise  | Query: What's the weather in London? | Tools Used: ['get_weather'] | Tool Calls: 1 | Answer Length: 49
Prompt: Concise  | Query: Tell me about Python programming | Tools Used: ['search_knowledge_base'] | Tool Calls: 1 | Answer Length: 41
Prompt: Verbose  | Query: What is 42 times 58?      | Tools Used: ['calculator'] | Tool Calls: 1 | Answer Length: 35
Prompt: Verbose  | Query: What's the weather in London? | Tools Used: ['get_weather'] | Tool Calls: 1 | Answer Length: 66
Prompt: Verbose  | Query: Tell me about Python programming | Tools Used: ['search_knowledge_base'] | Tool Calls: 1 | Answer Length: 2352
Prompt: Lazy     | Query: What is 42 times 58?      | Tools Used: [] | Tool Calls: 0 | Answer Length: 386
Prompt: Lazy     | Query: What's the weather in London? | Tools Used: ['get_weather'] | Tool Calls: 1 | Answer Length: 68
Prompt: Lazy     | Quer

### 🏗️ Exercise 8: Research & Report Agent

Build a two-phase agent pipeline:
1. **Phase 1 — Research:** Use a tool-calling agent to gather information (search knowledge base, calculate, etc.)
2. **Phase 2 — Report:** Feed the gathered information into a `with_structured_output()` call to produce a clean `ResearchReport` Pydantic object

**Requirements:**
- Define a `ResearchReport` model with: `title`, `summary`, `key_facts: List[str]`, `related_topics: List[str]`, `confidence_score: float`
- The research phase should actually call tools (search, etc.)
- The report phase should format the agent's findings into the structured model
- Test with: "What is RAG in machine learning?"

In [247]:
from pydantic import BaseModel, Field


class ResearchReport(BaseModel):
    """A structured research report."""

    title: str = Field(description="Title of the research report")
    summary: str = Field(description="A 2-3 sentence summary of findings")
    key_facts: list[str] = Field(description="3-5 key facts discovered")
    related_topics: list[str] = Field(description="Related topics worth exploring")
    confidence_score: float = Field(description="Confidence in findings, 0.0 to 1.0")


def research_and_report(topic: str) -> ResearchReport:
    """Two-phase pipeline.

    Phase 1: Use a tool-calling agent to research the topic
    Phase 2: Use structured output to format findings into a ResearchReport
    """
    research_agent = create_agent(
        model=ChatMistralAI(model="mistral-small-latest", temperature=0),
        tools=[search_knowledge_base, get_current_time],
    )
    response = research_agent.invoke({"messages": [("user", f"Research this topic thoroughly: {topic}")]})
    raw_findings = response["messages"][-1].content
    findings_text = raw_findings if isinstance(raw_findings, str) else str(raw_findings)

    report_model = ChatMistralAI(model="mistral-small-latest", temperature=0).with_structured_output(ResearchReport)
    report_result = report_model.invoke(
        f"Based on these research findings, create a structured report:\n\n{findings_text}",
    )

    if isinstance(report_result, ResearchReport):
        return report_result
    if isinstance(report_result, BaseModel):
        return ResearchReport.model_validate(report_result.model_dump())
    if isinstance(report_result, dict):
        return ResearchReport.model_validate(report_result)
    return ResearchReport.model_validate_json(str(report_result))


report = research_and_report("What is RAG in machine learning?")
print(f"📄 Title: {report.title}")
print(f"📝 Summary: {report.summary}")
print("🔑 Key Facts:")
for fact in report.key_facts:
    print(f"   • {fact}")
print(f"🔗 Related Topics: {report.related_topics}")
print(f"📊 Confidence: {report.confidence_score:.0%}")


📄 Title: Retrieval-Augmented Generation (RAG): Enhancing Language Models with Retrieval Mechanisms
📝 Summary: Retrieval-Augmented Generation (RAG) is a machine learning technique that combines retrieval-based methods with generative models to improve the performance of language models. By fetching relevant information from a large corpus before generating responses, RAG enhances accuracy and reduces hallucination. This approach is scalable and flexible, applicable in various NLP tasks such as question answering, dialogue systems, and summarization.
🔑 Key Facts:
   • RAG combines retrieval-based methods with generative models to improve language model performance.
   • The retrieval module fetches relevant documents using techniques like vector similarity search.
   • The generation module produces coherent responses based on retrieved information.
   • RAG enhances accuracy, reduces hallucination, and is scalable and flexible.
   • Applications include question answering, dialogue syst

### 💬 Exercise 9: Conversational Agent with Persistent Memory

Build a `PersistentChatAgent` that combines **tools** and **multi-turn memory** in a single class using LangGraph.

**Requirements:**
1. The agent should maintain conversation history across `.chat()` calls
2. It should have access to `calculator`, `get_weather`, and `save_note` / `get_notes` tools
3. Test a multi-turn scenario:
   - "What's 144 divided by 12?" → uses calculator
   - "Save that result as a note titled 'Math Result'" → uses save_note
   - "What's the weather in Tokyo?" → uses get_weather
   - "Save the Tokyo weather as a note too" → uses save_note
   - "Show me all my notes" → uses get_notes
   - "What was the first thing I asked you?" → must remember from conversation history

In [248]:
class PersistentChatAgent:
    """An agent that remembers the full conversation AND uses tools.

    Each call to .chat() appends to the running message history.
    """

    def __init__(self) -> None:
        """Initialize the agent with a model, tools, system prompt, and empty message history."""
        # Your code here:
        # 1. Initialize the ChatMistralAI model
        # 2. Define the tools list
        # 3. Create the LangGraph react agent
        # 4. Initialize an empty message history list
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.tools = [calculator, get_weather, search_knowledge_base, save_note, get_notes, get_current_time]
        self.agent = create_agent(
            model=self.model,
            tools=self.tools,
            system_prompt=(
                "You are a helpful assistant that remembers the entire conversation. "
                "Use tools whenever they can help you answer the user's questions. "
                "Maintain a running history of the conversation and use it to inform your answers."
            ),
        )
        self.messages: list = []  # This will store the full conversation history

    def chat(self, user_message: str) -> str:
        """Send a message, get a response, and maintain conversation history."""
        # Your code here:
        # 1. Append ("user", user_message) to self.messages
        # 2. Invoke the agent with the FULL message history
        # 3. Extract the final AI message content
        # 4. Append ("assistant", answer) to self.messages
        # 5. Return the answer
        self.messages.append(("user", user_message))
        response = self.agent.invoke({"messages": self.messages})
        final_msg = response["messages"][-1].content
        self.messages.append(("assistant", final_msg))
        return final_msg

    def get_history_length(self) -> int:
        """Get the number of messages in the conversation history."""
        # Return the number of messages
        return len(self.messages)


# Test the multi-turn scenario
notes_store.clear()  # Reset notes
agent = PersistentChatAgent()

conversation_flow = [
    "What's 144 divided by 12?",
    "Save that result as a note titled 'Math Result'",
    "What's the weather in Tokyo?",
    "Save the Tokyo weather as a note too",
    "Show me all my notes",
    "What was the first thing I asked you?",
]

for msg in conversation_flow:
    print(f"\n👤 {msg}")
    response = agent.chat(msg)
    print(f"🤖 {response}")
    print(f"   📊 History: {agent.get_history_length()} messages")




👤 What's 144 divided by 12?
🤖 The answer is 12.
   📊 History: 2 messages

👤 Save that result as a note titled 'Math Result'
🤖 I've saved the result as a note titled 'Math Result'.
   📊 History: 4 messages

👤 What's the weather in Tokyo?
🤖 The weather in Tokyo is currently clear with a temperature of 25°C.
   📊 History: 6 messages

👤 Save the Tokyo weather as a note too
🤖 What title would you like to give to this note?
   📊 History: 8 messages

👤 Show me all my notes
🤖 Here are your saved notes:

1. Math Result: 144 divided by 12 equals 12
2. Tokyo Weather: The weather in Tokyo is currently clear with a temperature of 25°C
   📊 History: 10 messages

👤 What was the first thing I asked you?
🤖 The first thing you asked me was "What's 144 divided by 12?"
   📊 History: 12 messages


### 🔀 Exercise 10: Agent Router — Classify & Dispatch

Build a system that **classifies** user queries and **routes** them to different specialized agents.

**Requirements:**
1. Create a `QueryClassifier` using `with_structured_output()` that classifies queries into:
   - `"math"` — send to a calculator agent
   - `"weather"` — send to a weather agent
   - `"knowledge"` — send to a knowledge base agent
   - `"general"` — answer directly without tools
2. Create a specialized mini-agent for each category
3. Build a `RouterAgent` that:
   - Classifies the incoming query
   - Dispatches to the right specialized agent
   - Returns the result along with which route was taken
4. Test with a variety of queries

In [ ]:
from enum import Enum

from pydantic import BaseModel, Field


class QueryCategory(str, Enum):
    """Categories for classifying user queries."""

    math = "math"
    weather = "weather"
    knowledge = "knowledge"
    general = "general"


class QueryClassification(BaseModel):
    """Classify a user query into the appropriate category."""

    category: QueryCategory = Field(description="The category this query belongs to")
    reasoning: str = Field(description="Brief explanation for the classification")


def _to_text(value: object) -> str:
    if isinstance(value, str):
        return value
    if hasattr(value, "content"):
        content = value.content
        if isinstance(content, str):
            return content
        return str(content)
    return str(value)


class RouterAgent:
    """Classifies incoming queries and routes them to specialized agents."""

    def __init__(self) -> None:
        """Initialize the router with a classification model and specialized agents for math, weather, and knowledge queries."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0)
        self.classifier = self.model.with_structured_output(QueryClassification)
        self.math_agent = create_agent(
            model=self.model,
            tools=[calculator],
            system_prompt="You are a math assistant. Use the calculator tool to solve math problems.",
        )
        self.weather_agent = create_agent(
            model=self.model,
            tools=[get_weather],
            system_prompt="You are a weather assistant. Use the get_weather tool to provide current weather information.",
        )
        self.knowledge_agent = create_agent(
            model=self.model,
            tools=[search_knowledge_base],
            system_prompt="You are a knowledge assistant. Use the search_knowledge_base tool to find information on various topics.",
        )

    def classify(self, query: str) -> QueryClassification:
        """Classify the query into a category using the classifier model."""
        result = self.classifier.invoke(
            f"Classify this query into one of: math, weather, knowledge, general. Query: {query}",
        )
        if isinstance(result, QueryClassification):
            return result
        if isinstance(result, BaseModel):
            return QueryClassification.model_validate(result.model_dump())
        if isinstance(result, dict):
            return QueryClassification.model_validate(result)
        return QueryClassification.model_validate_json(str(result))

    def route(self, query: str) -> dict[str, str]:
        """Classify the query and route it to the appropriate agent."""
        classification = self.classify(query)
        category = classification.category
        reasoning = classification.reasoning

        if category == QueryCategory.math:
            answer = _to_text(self.math_agent.invoke({"messages": [("user", query)]})["messages"][-1])
        elif category == QueryCategory.weather:
            answer = _to_text(self.weather_agent.invoke({"messages": [("user", query)]})["messages"][-1])
        elif category == QueryCategory.knowledge:
            answer = _to_text(self.knowledge_agent.invoke({"messages": [("user", query)]})["messages"][-1])
        else:
            answer = _to_text(self.model.invoke(query))

        return {"category": category.value, "reasoning": reasoning, "answer": answer}


test_queries = [
    "What is 256 divided by 8?",
    "What's the weather in San Francisco?",
    "Tell me about LangChain",
    "What are the benefits of exercise?",
    "Calculate 99 times 101",
    "Is it raining in London?",
]

router = RouterAgent()
for query in test_queries:
    result = router.route(query)
    print(f"\n👤 Query: {query}")
    print(f"   🏷️ Category: {result['category']} ({result['reasoning']})")
    print(f"   💬 Answer: {result['answer'][:150]}")



👤 Query: What is 256 divided by 8?
   🏷️ Category: math (The query involves a mathematical operation (division).)
   💬 Answer: The result of 256 divided by 8 is 32.

👤 Query: What's the weather in San Francisco?
   🏷️ Category: weather (The query asks for weather information, specifically about San Francisco.)
   💬 Answer: The weather in San Francisco is currently foggy with a temperature of 18°C.

👤 Query: Tell me about LangChain
   🏷️ Category: knowledge (The query 'Tell me about LangChain' is asking for information about a specific topic, which falls under the 'knowledge' category.)
   💬 Answer: LangChain is a framework designed to facilitate the development of applications powered by language models. It offers a range of tools and features th

👤 Query: What are the benefits of exercise?
   🏷️ Category: knowledge (The query asks for information about the benefits of exercise, which is a factual and informational question. It does not pertain to math, weather, or general topics.)
  

### 🌊 Exercise 11: Streaming Agent with Progress Callbacks

Build an agent wrapper that provides **real-time progress callbacks** during execution, so a user interface could display what the agent is doing.

**Requirements:**
1. Use LangGraph's `.stream()` method (shown in Part 4)
2. Create a `StreamingAgentRunner` class that:
   - Accepts callback functions: `on_thinking`, `on_tool_call`, `on_tool_result`, `on_final_answer`
   - Streams the agent execution and fires the callbacks at each step
   - Tracks total execution time and number of steps
3. Test with a query that triggers multiple tool calls
4. Implement simple callbacks that print progress with timestamps

In [250]:
from collections.abc import Callable


class StreamingAgentRunner:
    """Wraps a LangGraph agent and fires callbacks during execution."""

    def __init__(
        self,
        tools: list,
        on_thinking: Callable[[str], None] | None = None,
        on_tool_call: Callable[[str, dict], None] | None = None,
        on_tool_result: Callable[[str], None] | None = None,
        on_final_answer: Callable[[str], None] | None = None,
    ) -> None:
        """Initialize the runner with tools and optional callbacks."""
        self.agent = create_agent(
            model=ChatMistralAI(model="mistral-small-latest", temperature=0),
            tools=tools,
            system_prompt=(
                "You are a helpful assistant that explains your reasoning step by step. "
                "Use tools whenever they can help you answer the user's question."
            ),
        )
        self.on_thinking = on_thinking
        self.on_tool_call = on_tool_call
        self.on_tool_result = on_tool_result
        self.on_final_answer = on_final_answer
        self.steps = 0
        self.tools_used: set[str] = set()

    @staticmethod
    def _to_text(content: str | list[str | dict]) -> str:
        if isinstance(content, str):
            return content
        return str(content)

    @staticmethod
    def _extract_messages(node_output: dict[str, object]) -> list[BaseMessage]:
        """Extract message list from a node output."""
        messages = node_output.get("messages", [])
        return messages if isinstance(messages, list) else []

    def _handle_tool_calls(self, tool_calls: list[dict[str, object]]) -> None:
        """Handle tool call callbacks and tracking."""
        if self.on_thinking:
            self.on_thinking("choosing tool")

        for tool_call in tool_calls:
            tool_name = str(tool_call.get("name", "unknown"))
            tool_args = tool_call.get("args", {})
            self.tools_used.add(tool_name)

            if self.on_tool_call:
                self.on_tool_call(tool_name, tool_args if isinstance(tool_args, dict) else {})

    def _handle_message(self, msg: BaseMessage, final_answer: str) -> str:
        """Process one message and return updated final answer."""
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            self._handle_tool_calls(tool_calls)
            return final_answer

        if isinstance(msg, ToolMessage):
            if self.on_tool_result:
                self.on_tool_result(self._to_text(msg.content))
            return final_answer

        content = getattr(msg, "content", "")
        return self._to_text(content) if content else final_answer

    def run(self, user_message: str) -> dict[str, object]:
        """Stream agent execution with callbacks."""
        start_time = time.time()
        self.steps = 0
        self.tools_used.clear()
        final_answer = ""

        for step in self.agent.stream({"messages": [("user", user_message)]}):
            self.steps += 1
            for node_output in step.values():
                for msg in self._extract_messages(node_output):
                    final_answer = self._handle_message(msg, final_answer)

        if self.on_final_answer:
            self.on_final_answer(final_answer)

        return {
            "answer": final_answer,
            "steps": self.steps,
            "elapsed_seconds": time.time() - start_time,
            "tools_used": sorted(self.tools_used),
        }


def print_thinking(node_name: str) -> None:
    """Print a message when the agent is thinking in a node."""
    print(f"   ⏳ [{datetime.now(UTC).strftime('%H:%M:%S %Z')}] Agent thinking: {node_name}")


def print_tool_call(tool_name: str, tool_args: dict) -> None:
    """Print a message when the agent calls a tool."""
    print(f"   🔧 [{datetime.now(UTC).strftime('%H:%M:%S %Z')}] Calling: {tool_name}({tool_args})")


def print_tool_result(result: str) -> None:
    """Print a message when the agent receives a tool result."""
    print(f"   📋 [{datetime.now(UTC).strftime('%H:%M:%S %Z')}] Result: {result[:100]}")


def print_final(_answer: str) -> None:
    """Print a message when the agent produces the final answer."""
    print(f"   ✅ [{datetime.now(UTC).strftime('%H:%M:%S %Z')}] Final answer ready!")


runner = StreamingAgentRunner(
    tools=[calculator, get_weather, search_knowledge_base],
    on_thinking=print_thinking,
    on_tool_call=print_tool_call,
    on_tool_result=print_tool_result,
    on_final_answer=print_final,
)
print("🌊 Streaming agent execution:\n")
summary = runner.run("What is 25 * 17, and what's the weather in Paris?")
print(f"\n📊 Summary: {summary['steps']} steps in {summary['elapsed_seconds']:.1f}s")
print(f"   Tools used: {summary['tools_used']}")
print(f"   Answer: {summary['answer']}")


🌊 Streaming agent execution:

   ⏳ [17:20:54 UTC] Agent thinking: choosing tool
   🔧 [17:20:54 UTC] Calling: calculator({'operation': 'multiply', 'x': 25, 'y': 17})
   🔧 [17:20:54 UTC] Calling: get_weather({'city': 'Paris'})
   📋 [17:20:54 UTC] Result: 25.0 multiply 17.0 = 425.0
   📋 [17:20:54 UTC] Result: Weather in Paris: ☀️ 22°C, Sunny with light clouds
   ✅ [17:20:54 UTC] Final answer ready!

📊 Summary: 4 steps in 0.9s
   Tools used: ['calculator', 'get_weather']
   Answer: The result of 25 multiplied by 17 is 425. The weather in Paris is currently sunny with light clouds and a temperature of 22°C.


### 🤝 Exercise 12: Reflection Agent (Generate → Critique → Improve)

Build an agent that uses a **reflection loop** to iteratively improve its output. This is a common advanced pattern where one LLM call generates content and another critiques it.

**Requirements:**
1. Create a `ReflectionAgent` with two internal chains:
   - **Generator**: Writes content based on a user request
   - **Critic**: Reviews the generated content and provides feedback
2. The agent loops `max_rounds` times:
   - Generate → Critique → Revise based on critique → Critique again → ...
3. Stop early if the critic rates the content above a threshold (e.g., 8/10)
4. Use structured output for the critic: `CritiqueResult` with `score: int`, `feedback: str`, `suggestions: List[str]`
5. Test with: "Write a concise explanation of how neural networks learn"

In [251]:
from pydantic import BaseModel, Field


class CritiqueResult(BaseModel):
    """Structured critique of generated content."""

    score: int = Field(description="Quality score from 1 to 10")
    feedback: str = Field(description="Overall feedback on the content")
    suggestions: list[str] = Field(description="Specific suggestions for improvement")


class ReflectionAgent:
    """An agent that generates content and iteratively improves it using self-critique."""

    def __init__(self, max_rounds: int = 3, quality_threshold: int = 8) -> None:
        """Initialize the agent with a model, critic model, max rounds, quality threshold, and history."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0.3)
        self.critic_model = self.model.with_structured_output(CritiqueResult)
        self.max_rounds = max_rounds
        self.quality_threshold = quality_threshold
        self.history: list[dict] = []

    @staticmethod
    def _to_text(content: str | list[str | dict]) -> str:
        if isinstance(content, str):
            return content
        return str(content)

    def generate(self, request: str, previous_critique: str = "") -> str:
        """Generate or revise content based on the request and optional critique."""
        if not previous_critique:
            prompt = f"Generate concise, clear content for this request:\n\n{request}"
        else:
            prompt = (
                "Revise the content for this request based on the critique below. "
                "Apply the suggestions explicitly.\n\n"
                f"Critique:\n{previous_critique}\n\n"
                f"Request:\n{request}"
            )
        response = self.model.invoke(prompt)
        return self._to_text(response.content)

    def critique(self, content: str, original_request: str) -> CritiqueResult:
        """Critique the generated content."""
        prompt = (
            f"Rate and critique this content written for: '{original_request}'. "
            "Return score (1-10), feedback, and concrete suggestions.\n\n"
            f"Content:\n{content}"
        )
        result = self.critic_model.invoke(prompt)
        if isinstance(result, CritiqueResult):
            return result
        if isinstance(result, BaseModel):
            return CritiqueResult.model_validate(result.model_dump())
        if isinstance(result, dict):
            return CritiqueResult.model_validate(result)
        return CritiqueResult.model_validate_json(str(result))

    def run(self, request: str) -> dict[str, object]:
        """Run the full reflection loop."""
        self.history = []
        content = self.generate(request)

        for round_num in range(self.max_rounds):
            critique = self.critique(content, request)
            self.history.append(
                {
                    "round": round_num + 1,
                    "content": content,
                    "score": critique.score,
                    "feedback": critique.feedback,
                    "suggestions": critique.suggestions,
                },
            )
            print(f"Round {round_num + 1}: Score {critique.score}/10")

            if critique.score >= self.quality_threshold:
                break

            improvement_notes = critique.feedback
            if critique.suggestions:
                improvement_notes += "\n\nSuggestions:\n- " + "\n- ".join(critique.suggestions)
            content = self.generate(request, previous_critique=improvement_notes)

        return {
            "final_content": content,
            "rounds": len(self.history),
            "final_score": self.history[-1]["score"] if self.history else 0,
            "history": self.history,
        }


agent = ReflectionAgent(max_rounds=3, quality_threshold=8)
result = agent.run("Write a concise explanation of how neural networks learn")

print(f"\n{'='*70}")
print(f"📝 Final Content (after {result['rounds']} rounds, score: {result['final_score']}/10):\n")
print(result["final_content"])
print("\n📊 Improvement History:")
for i, round_info in enumerate(result["history"]):
    print(f"   Round {i+1}: Score {round_info['score']}/10 — {round_info['feedback'][:80]}...")


Round 1: Score 8/10

📝 Final Content (after 1 rounds, score: 8/10):

**How Neural Networks Learn**

Neural networks learn by adjusting their internal parameters (weights and biases) to minimize prediction errors. Here’s the process:

1. **Forward Pass**: Input data flows through the network, producing an output.
2. **Loss Calculation**: The output is compared to the true value using a loss function (e.g., mean squared error).
3. **Backpropagation**: The network calculates how much each weight contributed to the error and adjusts them in the opposite direction (gradient descent).
4. **Iteration**: Steps 1–3 repeat with new data until the network’s predictions improve.

Over time, the network refines its weights to make accurate predictions.

📊 Improvement History:
   Round 1: Score 8/10 — The content is clear and concise, effectively explaining the basic process of ho...


### 🏗️ Exercise 13: Hierarchical Multi-Agent System (Planner + Executors)

Build a **two-layer agent architecture** where a **Planner** agent decomposes a complex task into sub-tasks and dispatches them to **Specialist Executor** agents.

**Requirements:**
1. Define a structured `Plan` model with a list of `SubTask` items (each with `agent`, `instruction`, `depends_on`)
2. Create three specialist agents as simple functions or classes:
   - **ResearcherAgent**: Answers factual questions (uses `search_knowledge_base` or a stub)
   - **AnalystAgent**: Performs numerical analysis / comparisons (uses `calculator` or logic)
   - **WriterAgent**: Synthesises findings into polished prose
3. Build a `PlannerAgent` that:
   - Receives a high-level goal from the user
   - Returns a structured `Plan` using `with_structured_output`
4. Build an `OrchestratorAgent` that:
   - Calls `PlannerAgent` to get the plan
   - Executes sub-tasks in dependency order (sequential for simplicity)
   - Passes the output of each step as context to dependent steps
   - Returns the final writer output and an execution log
5. Test with: `"Compare supervised vs unsupervised learning and write a short summary"`


In [253]:
from pydantic import BaseModel, Field


class SubTask(BaseModel):
    """A single step in the execution plan."""

    id: str = Field(description="Unique identifier, e.g. 'step_1'")
    agent: str = Field(
        description="Which agent to use: 'researcher', 'analyst', or 'writer'",
    )
    instruction: str = Field(description="Detailed instruction for the agent")
    depends_on: list[str] = Field(
        default_factory=list,
        description="List of step ids whose output this step needs as input",
    )


class Plan(BaseModel):
    """The full execution plan produced by the Planner."""

    goal: str = Field(description="The high-level user goal")
    steps: list[SubTask] = Field(description="Ordered list of sub-tasks to execute")


def _extract_text(response: object) -> str:
    """Convert model output to plain text."""
    if isinstance(response, str):
        return response
    if isinstance(response, BaseMessage):
        content = response.content
        if isinstance(content, str):
            return content
        return str(content)
    return str(response)


class ResearcherAgent:
    """Answers factual questions by querying the LLM directly."""

    def __init__(self) -> None:
        """Initialize the researcher model."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0.1)

    def run(self, instruction: str, context: str = "") -> str:
        """Return a factual answer."""
        prompt = (
            "You are a research assistant.\n\n"
            f"Context:\n{context or 'None'}\n\n"
            f"Instruction: {instruction}\n\n"
            "Answer:"
        )
        response = self.model.invoke(prompt)
        return _extract_text(response)


class AnalystAgent:
    """Performs analysis, comparisons and reasoning over provided information."""

    def __init__(self) -> None:
        """Initialize the analyst model."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0.1)

    def run(self, instruction: str, context: str = "") -> str:
        """Return structured analytical insights."""
        prompt = (
            "You are an analytical assistant.\n\n"
            f"Context:\n{context or 'None'}\n\n"
            f"Instruction: {instruction}\n\n"
            "Analysis:"
        )
        response = self.model.invoke(prompt)
        return _extract_text(response)


class WriterAgent:
    """Synthesises research and analysis into polished prose."""

    def __init__(self) -> None:
        """Initialize the writer model."""
        self.model = ChatMistralAI(model="mistral-small-latest", temperature=0.7)

    def run(self, instruction: str, context: str = "") -> str:
        """Return a well-written piece of text."""
        prompt = (
            "You are a skilled technical writer.\n\n"
            f"Context:\n{context or 'None'}\n\n"
            f"Instruction: {instruction}\n\n"
            "Write:"
        )
        response = self.model.invoke(prompt)
        return _extract_text(response)


class PlannerAgent:
    """Decomposes a high-level goal into an ordered list of specialist sub-tasks."""

    def __init__(self) -> None:
        """Initialize the planner model with structured output."""
        self.model = ChatMistralAI(
            model="mistral-small-latest", temperature=0.1,
        ).with_structured_output(Plan)

    def plan(self, goal: str) -> Plan:
        """Generate a Plan for the given goal."""
        system_prompt = (
            "You are a planner agent that breaks down a high-level goal into a structured "
            "execution plan. Available agents: "
            "'researcher' (facts), 'analyst' (reasoning/comparison), "
            "'writer' (final synthesis). "
            "Each step must include id, agent, instruction, and depends_on."
        )
        messages = [SystemMessage(content=system_prompt), HumanMessage(content=goal)]
        raw_plan = self.model.invoke(messages)

        if isinstance(raw_plan, Plan):
            return raw_plan
        if isinstance(raw_plan, BaseModel):
            return Plan.model_validate(raw_plan.model_dump())
        if isinstance(raw_plan, dict):
            return Plan.model_validate(raw_plan)

        msg = "Planner returned an unsupported output type."
        raise TypeError(msg)


class OrchestratorAgent:
    """Coordinates planning and execution across specialist agents."""

    def __init__(self) -> None:
        """Initialize all agents."""
        self.planner = PlannerAgent()
        self.researcher = ResearcherAgent()
        self.analyst = AnalystAgent()
        self.writer = WriterAgent()

    def _get_agent(self, agent_name: str) -> ResearcherAgent | AnalystAgent | WriterAgent:
        """Return the specialist agent by name."""
        if agent_name == "researcher":
            return self.researcher
        if agent_name == "analyst":
            return self.analyst
        if agent_name == "writer":
            return self.writer
        msg = f"Unknown agent: {agent_name}"
        raise ValueError(msg)

    def _resolve_context(self, depends_on: list[str], results: dict[str, str]) -> str:
        """Build context from dependency outputs."""
        parts: list[str] = []
        for step_id in depends_on:
            if step_id not in results:
                msg = f"Missing dependency output for step '{step_id}'."
                raise ValueError(msg)
            parts.append(f"=== {step_id} output ===\n{results[step_id]}")
        return "\n\n".join(parts)

    def run(self, goal: str) -> dict[str, object]:
        """Execute the full multi-agent pipeline."""
        plan = self.planner.plan(goal)

        results: dict[str, str] = {}
        log: list[str] = []

        for step in plan.steps:
            context = self._resolve_context(step.depends_on, results)
            agent = self._get_agent(step.agent)
            output = agent.run(step.instruction, context)
            results[step.id] = output
            log.append(f"Step {step.id} output: {output[:100]}...")

        writer_step_id = next(
            (s.id for s in reversed(plan.steps) if s.agent == "writer" and s.id in results),
            None,
        )
        final_output = results[writer_step_id] if writer_step_id else list(results.values())[-1]

        return {
            "goal": goal,
            "plan": plan,
            "results": results,
            "final_output": final_output,
            "log": log,
        }


orchestrator = OrchestratorAgent()
output = orchestrator.run(
    "Compare supervised vs unsupervised learning and write a short summary",
)

plan_result = output.get("plan")
plan_steps = plan_result.steps if isinstance(plan_result, Plan) else []

print("\n" + "=" * 70)
print(f"🎯 Goal: {output['goal']}")
print(f"\n📋 Plan ({len(plan_steps)} steps):")
for step in plan_steps:
    deps = f" (depends on: {step.depends_on})" if step.depends_on else ""
    print(f"  [{step.id}] {step.agent.upper()}: {step.instruction[:60]}...{deps}")

print("\n📝 Execution Log:")
log_entries = output.get("log", [])
if isinstance(log_entries, list):
    for entry in log_entries:
        print(f"  {entry}")

print("\n✍️  Final Output:\n")
print(output["final_output"])



🎯 Goal: Compare supervised vs unsupervised learning and write a short summary

📋 Plan (4 steps):
  [step_1] RESEARCHER: Find key characteristics of supervised learning...
  [step_2] RESEARCHER: Find key characteristics of unsupervised learning...
  [step_3] ANALYST: Compare and contrast the characteristics of supervised and u... (depends on: ['step_1', 'step_2'])
  [step_4] WRITER: Write a short summary comparing supervised and unsupervised ... (depends on: ['step_3'])

📝 Execution Log:
  Step step_1 output: Supervised learning is a type of machine learning where a model is trained on a labeled dataset, mea...
  Step step_2 output: Unsupervised learning is a type of machine learning where the model learns patterns and structures f...
  Step step_3 output: Here’s a structured comparison and contrast of **supervised** and **unsupervised learning** based on...
  Step step_4 output: **Summary: Supervised vs. Unsupervised Learning**

Supervised and unsupervised learning are two core...

✍️